# Téléchargement des embeddings de mots

Embeddings utilisés pour les modèles de deep learning (Tâche 6).

- **GloVe Twitter 200d** : pré-entraîné sur 27 milliards de tweets. Choisi pour la correspondance du domaine.
- **FastText 300d** : pré-entraîné sur Wikipedia et actualités. Gère les mots hors-vocabulaire via les sous-mots (abréviations, argot des tweets).

Destination : `data/embeddings/` (ignoré par git).

In [1]:
import os
import sys
import ssl
import zipfile
import urllib.request

import gensim.downloader as gensim_dl

# Contournement SSL pour macOS
ssl._create_default_https_context = ssl._create_unverified_context

EMBEDDINGS_DIR = os.path.abspath(os.path.join("..", "data", "embeddings"))
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
print(f"Répertoire des embeddings : {EMBEDDINGS_DIR}")

Repertoire des embeddings : /Users/arthur/code/OC2/Projet7/data/embeddings


## 1. GloVe Twitter 200d

**Source :** Stanford NLP - https://nlp.stanford.edu/data/glove.twitter.27B.zip

Entraîné sur 27 milliards de tokens Twitter. Le vocabulaire, les abréviations et le style informel propres aux tweets sont bien représentés.

**Dimension :** 200 (compromis entre qualité sémantique et empreinte mémoire).

In [2]:
GLOVE_ZIP_URL = "https://nlp.stanford.edu/data/glove.twitter.27B.zip"
GLOVE_ZIP_PATH = os.path.join(EMBEDDINGS_DIR, "glove.twitter.27B.zip")
GLOVE_FILE = "glove.twitter.27B.200d.txt"
GLOVE_PATH = os.path.join(EMBEDDINGS_DIR, GLOVE_FILE)


def download_with_progress(url: str, dest_path: str) -> None:
    def reporthook(block_num, block_size, total_size):
        downloaded = block_num * block_size
        if total_size > 0:
            percent = min(downloaded * 100 / total_size, 100)
            mb_downloaded = downloaded / 1_048_576
            mb_total = total_size / 1_048_576
            print(
                f"\r  {percent:5.1f}%  ({mb_downloaded:.1f} / {mb_total:.1f} Mo)",
                end="",
                flush=True,
            )

    print(f"Téléchargement de {url}")
    urllib.request.urlretrieve(url, dest_path, reporthook=reporthook)
    print()


if os.path.exists(GLOVE_PATH):
    print(f"Fichier déjà présent : {GLOVE_PATH}")
    print("Téléchargement ignoré.")
else:
    if not os.path.exists(GLOVE_ZIP_PATH):
        download_with_progress(GLOVE_ZIP_URL, GLOVE_ZIP_PATH)
    else:
        print(f"Archive déjà présente : {GLOVE_ZIP_PATH}")

    # Extraction du fichier 200d uniquement (l'archive contient aussi 25d, 50d, 100d)
    print(f"Extraction de {GLOVE_FILE} ...")
    with zipfile.ZipFile(GLOVE_ZIP_PATH, "r") as zf:
        zf.extract(GLOVE_FILE, EMBEDDINGS_DIR)
    print(f"Extraction terminée : {GLOVE_PATH}")

    os.remove(GLOVE_ZIP_PATH)
    print("Archive ZIP supprimée.")

Telechargement de https://nlp.stanford.edu/data/glove.twitter.27B.zip
  100.0%  (1450.0 / 1450.0 Mo)
Extraction de glove.twitter.27B.200d.txt ...
Extraction terminee : /Users/arthur/code/OC2/Projet7/data/embeddings/glove.twitter.27B.200d.txt
Archive ZIP supprimee.


### Vérification de GloVe Twitter 200d

Format attendu : un mot suivi de 200 valeurs flottantes par ligne.

In [3]:
def verify_glove(filepath: str, n_lines_preview: int = 3) -> None:
    print(f"Vérification de : {filepath}")
    file_size_mb = os.path.getsize(filepath) / 1_048_576
    print(f"  Taille du fichier : {file_size_mb:.1f} Mo")

    num_vectors = 0
    embedding_dim = None

    with open(filepath, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            parts = line.rstrip().split(" ")
            word = parts[0]
            values = parts[1:]

            if embedding_dim is None:
                embedding_dim = len(values)

            if i < n_lines_preview:
                preview = " ".join(values[:5])
                print(f"  Ligne {i+1} | mot : '{word}' | début vecteur : [{preview} ...]")

            num_vectors += 1

    print(f"\n  Nombre de vecteurs : {num_vectors:,}")
    print(f"  Dimension des embeddings : {embedding_dim}")


verify_glove(GLOVE_PATH)

Verification de : /Users/arthur/code/OC2/Projet7/data/embeddings/glove.twitter.27B.200d.txt
  Taille du fichier : 1962.3 Mo
  Ligne 1 | mot : '<user>' | debut vecteur : [0.31553 0.53765 0.10177 0.032553 0.003798 ...]
  Ligne 2 | mot : '.' | debut vecteur : [0.35132 0.00056084 -0.21488 -0.04707 -0.17777 ...]
  Ligne 3 | mot : ':' | debut vecteur : [0.80767 0.49786 0.082696 -0.0079298 0.082471 ...]

  Nombre de vecteurs : 1,193,514
  Dimension des embeddings : 200


## 2. FastText 300d

**Source :** Facebook AI Research via `gensim.downloader` - modèle `fasttext-wiki-news-subwords-300`

FastText représente chaque mot comme une somme de vecteurs de sous-mots (n-grammes de caractères), ce qui permet de générer un vecteur même pour les mots hors-vocabulaire. Utile pour les tweets (mots inventés, abréviations, fautes d'orthographe).

Entraîné sur Wikipedia et les actualités en anglais (1 million de mots).

**Dimension :** 300.

In [4]:
FASTTEXT_MODEL_NAME = "fasttext-wiki-news-subwords-300"
FASTTEXT_PATH = os.path.join(EMBEDDINGS_DIR, f"{FASTTEXT_MODEL_NAME}.vec")

if os.path.exists(FASTTEXT_PATH):
    print(f"Fichier déjà présent : {FASTTEXT_PATH}")
    print("Téléchargement ignoré.")
else:
    print(f"Téléchargement du modèle '{FASTTEXT_MODEL_NAME}' via gensim.downloader ...")
    print("(Peut prendre plusieurs minutes selon la connexion.)")
    ft_model = gensim_dl.load(FASTTEXT_MODEL_NAME)
    print("Téléchargement terminé.")

    print(f"Sauvegarde vers : {FASTTEXT_PATH}")
    ft_model.save_word2vec_format(FASTTEXT_PATH)
    print("Sauvegarde terminée.")

Telechargement du modele 'fasttext-wiki-news-subwords-300' via gensim.downloader ...
(Cette operation peut prendre plusieurs minutes selon la connexion.)
[==================================================] 100.0% 958.5/958.4MB downloaded
Telechargement termine.
Sauvegarde vers : /Users/arthur/code/OC2/Projet7/data/embeddings/fasttext-wiki-news-subwords-300.vec
Sauvegarde terminee.


### Vérification de FastText 300d

Format Word2Vec texte : la première ligne donne nombre de vecteurs et dimension, puis un mot suivi de ses valeurs par ligne.

In [5]:
def verify_word2vec_format(filepath: str, n_lines_preview: int = 3) -> None:
    print(f"Vérification de : {filepath}")
    file_size_mb = os.path.getsize(filepath) / 1_048_576
    print(f"  Taille du fichier : {file_size_mb:.1f} Mo")

    with open(filepath, "r", encoding="utf-8") as f:
        header = f.readline().strip().split()
        num_vectors, embedding_dim = int(header[0]), int(header[1])
        print(f"  Nombre de vecteurs : {num_vectors:,}")
        print(f"  Dimension des embeddings : {embedding_dim}")

        print(f"\n  Aperçu des {n_lines_preview} premiers vecteurs :")
        for i in range(n_lines_preview):
            line = f.readline().rstrip()
            parts = line.split(" ")
            word = parts[0]
            preview = " ".join(parts[1:6])
            print(f"  Ligne {i+2} | mot : '{word}' | début vecteur : [{preview} ...]")


verify_word2vec_format(FASTTEXT_PATH)

Verification de : /Users/arthur/code/OC2/Projet7/data/embeddings/fasttext-wiki-news-subwords-300.vec
  Taille du fichier : 2755.6 Mo
  Nombre de vecteurs : 999,999
  Dimension des embeddings : 300

  Apercu des 3 premiers vecteurs :
  Ligne 2 | mot : ',' | debut vecteur : [0.020344 -0.012294 -0.0075729 0.018694 0.017297 ...]
  Ligne 3 | mot : 'the' | debut vecteur : [0.024249 0.0048235 0.018411 0.011867 0.019167 ...]
  Ligne 4 | mot : '.' | debut vecteur : [0.0048517 -0.0029997 0.067174 0.013612 -0.062353 ...]


## Récapitulatif des fichiers téléchargés

In [6]:
print("État des fichiers dans", EMBEDDINGS_DIR)
print("-" * 60)

files_info = [
    (GLOVE_PATH, "GloVe Twitter 200d"),
    (FASTTEXT_PATH, "FastText Wiki News 300d"),
]

all_present = True
for path, label in files_info:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1_048_576
        print(f"  [OK] {label}")
        print(f"       {os.path.basename(path)}  ({size_mb:.1f} Mo)")
    else:
        print(f"  [MANQUANT] {label}")
        print(f"       {os.path.basename(path)}")
        all_present = False

print("-" * 60)
if all_present:
    print("Tous les embeddings sont disponibles. Prêt pour la Tâche 6.")
else:
    print("Certains fichiers sont manquants. Relancer les cellules correspondantes.")

Etat des fichiers dans /Users/arthur/code/OC2/Projet7/data/embeddings
------------------------------------------------------------
  [OK] GloVe Twitter 200d
       glove.twitter.27B.200d.txt  (1962.3 Mo)
  [OK] FastText Wiki News 300d
       fasttext-wiki-news-subwords-300.vec  (2755.6 Mo)
------------------------------------------------------------
Tous les embeddings sont disponibles. Pret pour la Tache 6.
